In [1]:
!pip install ultralytics

In [6]:
import os
from ultralytics import YOLO

dataset_path = "./safety_helmet_dataset/"
images_path = os.path.join(dataset_path, "images")
annotation_path = os.path.join(dataset_path, "annotations")

In [9]:
import xml.etree.ElementTree as ET

classes = ["helmet", "head", "person"] 

def convert_bbox(size, box):
    dw, dh = 1./size[0], 1./size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return x*dw, y*dh, w*dw, h*dh

def replace_xml_with_txt(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)
    
    filename = root.find("filename").text
    file_id = os.path.splitext(filename)[0]
    
    txt_path = xml_file.replace('.xml', '.txt')
    with open(txt_path, "w") as f:
        for obj in root.iter("object"):
            cls = obj.find("name").text
            if cls not in classes:
                continue
            cls_id = classes.index(cls)
            
            xmlbox = obj.find("bndbox")
            b = (
                float(xmlbox.find("xmin").text),
                float(xmlbox.find("xmax").text),
                float(xmlbox.find("ymin").text),
                float(xmlbox.find("ymax").text),
            )
            bb = convert_bbox((w, h), b)
            f.write(f"{cls_id} {' '.join(f'{coord:.6f}' for coord in bb)}\n")
    
    os.remove(xml_file)

total = 0
for xml_file in os.listdir(annotation_path):
    if xml_file.endswith(".xml"):
        replace_xml_with_txt(os.path.join(annotation_path, xml_file))
        total += 1

print(f"\n{total} XML → TXT")


5000 XML → TXT


In [11]:
from sklearn.model_selection import train_test_split
import shutil

train_img_dir = "./safety_helmet_dataset/images/train"
val_img_dir = "./safety_helmet_dataset/images/val"
train_lbl_dir = "./safety_helmet_dataset/annotations/train" 
val_lbl_dir = "./safety_helmet_dataset/annotations/val"

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

image_files = [f for f in os.listdir("./safety_helmet_dataset/images") if f.endswith('.png')]
txt_files = [f.replace('.png', '.txt') for f in image_files]

train_imgs, val_imgs = train_test_split(image_files, test_size=0.2, random_state=42)

for img in train_imgs:
    shutil.move(f"./safety_helmet_dataset/images/{img}", f"{train_img_dir}/{img}")
    shutil.move(f"./safety_helmet_dataset/annotations/{img.replace('.png','.txt')}", f"{train_lbl_dir}/{img.replace('.png','.txt')}")

for img in val_imgs:
    shutil.move(f"./safety_helmet_dataset/images/{img}", f"{val_img_dir}/{img}")
    shutil.move(f"./safety_helmet_dataset/annotations/{img.replace('.png','.txt')}", f"{val_lbl_dir}/{img.replace('.png','.txt')}")

In [16]:
data_yaml = """\
path: ./safety_helmet_dataset
train: images/train
val: images/val
nc: 3
names: 
  0: helmet
  1: head  
  2: person
"""

with open("./safety_helmet_dataset/data.yaml", "w") as f:
    f.write(data_yaml)

In [18]:
model = YOLO("yolov8n.pt")
results = model.train(
    data="./safety_helmet_dataset/data.yaml",
    epochs=70,
    imgsz=640,
    batch=16,
    name="helmet_detector_perfect"
)

New https://pypi.org/project/ultralytics/8.4.32 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.30 🚀 Python-3.13.9 torch-2.11.0 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./safety_helmet_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_detector_perfect, n